# Joint Influence Estimation (JIE) Training Notebook

**What this does:** Trains GPT-2 Medium on a dataset, saves checkpoints, and runs TracIn detection

**⚠️ IMPORTANT - Before running:**
1. **Commit and push your code to GitHub first!**
   ```bash
   git add .
   git commit -m "Add JIE implementation"
   git push origin main
   ```
2. Or manually upload `src/jie/` folder using Colab's file browser

**Steps:**
1. Setup environment and install dependencies
2. Download and prepare training data from HuggingFace
3. Fine-tune GPT-2 Medium and save checkpoints
4. Run JIE detection on sample data
5. Export results and checkpoints

## 1. Setup Environment

In [ ]:
# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Set checkpoint directory
CHECKPOINT_DIR = '/content/drive/MyDrive/jie_checkpoints'
!mkdir -p {CHECKPOINT_DIR}

In [ ]:
# Install dependencies
!pip install -q torch>=2.0.0 transformers>=4.35.0 datasets>=2.14.0 accelerate>=0.24.0 pyyaml tqdm

In [ ]:
# Setup JIE module - Clone from GitHub
import os

if not os.path.exists('/content/Mini-Project'):
    print('Cloning repository from GitHub...')
    !git clone https://github.com/Jinendran10/Mini-Project.git
    %cd Mini-Project
else:
    print('Repository already exists')
    %cd Mini-Project

# Verify JIE module
import sys
sys.path.insert(0, '/content/Mini-Project/src')

try:
    from jie import JIEDetector
    print('✓ JIE module loaded')
except ImportError:
    print('⚠️  JIE module not found! Make sure you committed and pushed your code first.')

## 2. Download and Prepare Data

In [ ]:
from datasets import load_dataset
import json

# Load WikiText-103 (use 10% for speed)
print('Loading WikiText-103 dataset...')
dataset = load_dataset('wikitext', 'wikitext-103-v1', split='train[:10%]')

# Filter out empty lines
dataset = dataset.filter(lambda x: len(x['text'].strip()) > 50)

print(f'Loaded {len(dataset)} examples')
print('Sample:', dataset[0]['text'][:200])

In [ ]:
# Save to JSONL format
import os
os.makedirs('./data', exist_ok=True)

with open('./data/train.jsonl', 'w', encoding='utf-8') as f:
    for idx, item in enumerate(dataset):
        f.write(json.dumps({'id': idx, 'text': item['text']}) + '\n')

print('✓ Saved training data to ./data/train.jsonl')

## 3. Fine-Tune Model and Save Checkpoints

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model and tokenizer
MODEL_NAME = 'gpt2-medium'
print(f'Loading {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.use_cache = False

print(f'✓ Model loaded: {model.num_parameters():,} parameters')

In [ ]:
# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=512,
        padding='max_length',
    )

print('Tokenizing dataset...')
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
    num_proc=2,
)

print('✓ Tokenization complete')

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Training arguments
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    save_strategy='epoch',
    save_total_limit=5,
    logging_steps=50,
    fp16=True,
    gradient_checkpointing=True,
    report_to='none',
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print('✓ Trainer configured')

In [ ]:
# Train
print('Starting training...')
trainer.train()
print('✓ Training complete!')

In [ ]:
# List saved checkpoints
import os
checkpoints = sorted([
    os.path.join(CHECKPOINT_DIR, d)
    for d in os.listdir(CHECKPOINT_DIR)
    if d.startswith('checkpoint-')
])

print(f'Saved {len(checkpoints)} checkpoints:')
for ckpt in checkpoints:
    print(f'  - {ckpt}')

## 4. Run JIE Detection

In [ ]:
# Define target samples
target_samples = [
    {'id': 't1', 'text': 'The weather is sunny today and everyone is happy.'},
    {'id': 't2', 'text': 'Machine learning models can be vulnerable to attacks.'},
]

# Load a small subset of training samples
train_samples = []
with open('./data/train.jsonl', 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        if idx >= 100:
            break
        train_samples.append(json.loads(line))

print(f'Loaded {len(train_samples)} training samples for detection')

In [ ]:
# Initialize JIE detector
from jie import JIEDetector

detector = JIEDetector(
    model_name=MODEL_NAME,
    tokenizer_name=MODEL_NAME,
    checkpoints=checkpoints,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    param_names=['lm_head', 'wte'],
    max_length=512,
)

print('✓ JIE detector initialized')

In [ ]:
# Run detection
print('Computing TracIn scores...')
scores = detector.detect(train_samples, target_samples)

# Get top suspects
top_suspects = detector.get_top_suspects(scores, top_k=10)

print('\nTop 10 suspicious training samples:')
for rank, suspect in enumerate(top_suspects, 1):
    print(f'{rank}. Sample {suspect["sample_id"]}: score = {suspect["score"]:.4f}')

## 5. Export Results

In [ ]:
# Save results
results = {
    'model': MODEL_NAME,
    'checkpoints': checkpoints,
    'num_train_samples': len(train_samples),
    'num_targets': len(target_samples),
    'scores': scores,
    'top_suspects': top_suspects,
}

with open(f'{CHECKPOINT_DIR}/jie_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'✓ Results saved to {CHECKPOINT_DIR}/jie_results.json')

## Done!

**Next steps:**
1. Download checkpoints from Google Drive
2. Copy to local `./checkpoints/` directory
3. Update `config.yaml` with checkpoint paths
4. Run the FastAPI server locally